
# TorchRL trainer: A DQN example
**Author**: [Vincent Moens](https://github.com/vmoens)



TorchRL provides a generic :class:`~torchrl.trainers.Trainer` class to handle
your training loop. The trainer executes a nested loop where the outer loop
is the data collection and the inner loop consumes this data or some data
retrieved from the replay buffer to train the model.
At various points in this training loop, hooks can be attached and executed at
given intervals.

In this tutorial, we will be using the trainer class to train a DQN algorithm
to solve the CartPole task from scratch.

Main takeaways:

- Building a trainer with its essential components: data collector, loss
  module, replay buffer and optimizer.
- Adding hooks to a trainer, such as loggers, target network updaters and such.

The trainer is fully customisable and offers a large set of functionalities.
The tutorial is organised around its construction.
We will be detailing how to build each of the components of the library first,
and then put the pieces together using the :class:`~torchrl.trainers.Trainer`
class.

Along the road, we will also focus on some other aspects of the library:

- how to build an environment in TorchRL, including transforms (e.g. data
  normalization, frame concatenation, resizing and turning to grayscale)
  and parallel execution. Unlike what we did in the
  `DDPG tutorial <coding_ddpg>`, we
  will normalize the pixels and not the state vector.
- how to design a :class:`~torchrl.modules.QValueActor` object, i.e. an actor
  that estimates the action values and picks up the action with the highest
  estimated return;
- how to collect data from your environment efficiently and store them
  in a replay buffer;
- how to use multi-step, a simple preprocessing step for off-policy algorithms;
- and finally how to evaluate your model.

**Prerequisites**: We encourage you to get familiar with torchrl through the
`PPO tutorial <coding_ppo>` first.

## DQN

DQN ([Deep Q-Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf)) was
the founding work in deep reinforcement learning.

On a high level, the algorithm is quite simple: Q-learning consists in
learning a table of state-action values in such a way that, when
encountering any particular state, we know which action to pick just by
searching for the one with the highest value. This simple setting
requires the actions and states to be
discrete, otherwise a lookup table cannot be built.

DQN uses a neural network that encodes a map from the state-action space to
a value (scalar) space, which amortizes the cost of storing and exploring all
the possible state-action combinations: if a state has not been seen in the
past, we can still pass it in conjunction with the various actions available
through our neural network and get an interpolated value for each of the
actions available.

We will solve the classic control problem of the cart pole. From the
Gymnasium doc from where this environment is retrieved:

| A pole is attached by an un-actuated joint to a cart, which moves along a
| frictionless track. The pendulum is placed upright on the cart and the goal
| is to balance the pole by applying forces in the left and right direction
| on the cart.

.. figure:: /_static/img/cartpole_demo.gif
   :alt: Cart Pole

We do not aim at giving a SOTA implementation of the algorithm, but rather
to provide a high-level illustration of TorchRL features in the context
of this algorithm.



In [1]:
import os

import gymnasium

import torch
from tensordict.nn import TensorDictSequential
from torch import nn
from torchrl.collectors import MultiaSyncDataCollector, SyncDataCollector
from torchrl.data import LazyMemmapStorage, MultiStep, TensorDictReplayBuffer, LazyTensorStorage
from torchrl.envs import (
    EnvCreator,
    ExplorationType,
    ParallelEnv,
    RewardScaling,
    StepCounter,
    DTypeCastTransform,
    DoubleToFloat, check_env_specs, ActionMask,
)
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.transforms import (
    CatFrames,
    Compose,
    GrayScale,
    ObservationNorm,
    Resize,
    ToTensorImage,
    TransformedEnv,
)
from torchrl.modules import DuelingCnnDQNet, EGreedyModule, QValueActor

from torchrl.objectives import DQNLoss, SoftUpdate
from torchrl.record.loggers import TensorboardLogger
from torchrl.trainers import (
    LogScalar,
    LogValidationReward,
    ReplayBufferTrainer,
    Trainer,
    UpdateWeights,
)

from train.models import MLP

Let's get started with the various pieces we need for our algorithm:

- An environment;
- A policy (and related modules that we group under the "model" umbrella);
- A data collector, which makes the policy play in the environment and
  delivers training data;
- A replay buffer to store the training data;
- A loss module, which computes the objective function to train our policy
  to maximise the return;
- An optimizer, which performs parameter updates based on our loss.

Additional modules include a logger, a recorder (executes the policy in
"eval" mode) and a target network updater. With all these components into
place, it is easy to see how one could misplace or misuse one component in
the training script. The trainer is there to orchestrate everything for you!

## Building the environment

First let's write a helper function that will output an environment. As usual,
the "raw" environment may be too simple to be used in practice and we'll need
some data transformation to expose its output to the policy.

We will be using five transforms:

- :class:`~torchrl.envs.StepCounter` to count the number of steps in each trajectory;
- :class:`~torchrl.envs.transforms.ToTensorImage` will convert a ``[W, H, C]`` uint8
  tensor in a floating point tensor in the ``[0, 1]`` space with shape
  ``[C, W, H]``;
- :class:`~torchrl.envs.transforms.RewardScaling` to reduce the scale of the return;
- :class:`~torchrl.envs.transforms.GrayScale` will turn our image into grayscale;
- :class:`~torchrl.envs.transforms.Resize` will resize the image in a 64x64 format;
- :class:`~torchrl.envs.transforms.CatFrames` will concatenate an arbitrary number of
  successive frames (``N=4``) in a single tensor along the channel dimension.
  This is useful as a single image does not carry information about the
  motion of the cartpole. Some memory about past observations and actions
  is needed, either via a recurrent neural network or using a stack of
  frames.
- :class:`~torchrl.envs.transforms.ObservationNorm` which will normalize our observations
  given some custom summary statistics.

In practice, our environment builder has two arguments:

- ``parallel``: determines whether multiple environments have to be run in
  parallel. We stack the transforms after the
  :class:`~torchrl.envs.ParallelEnv` to take advantage
  of vectorization of the operations on device, although this would
  technically work with every single environment attached to its own set of
  transforms.
- ``obs_norm_sd`` will contain the normalizing constants for
  the :class:`~torchrl.envs.ObservationNorm` transform.




In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
from env.battleship_env import BattleshipEnv
gymnasium.register(
    id="Battleship-v0",
    entry_point=BattleshipEnv,
)

In [3]:
def env_maker():
    from env.battleship_env import BattleshipEnv
    gymnasium.register(
        id="Battleship-v0",
        entry_point=BattleshipEnv,
    )

    return TransformedEnv(
        GymEnv("Battleship-v0", categorical_action_encoding=True, device=device),
        Compose(
            # normalize observations
            # ObservationNorm(in_keys=["observation"]),
            DTypeCastTransform(
                torch.int8, torch.float32, in_keys=['observation']
            ),
            DTypeCastTransform(
                torch.int8, torch.bool, in_keys=['mask']
            ),
            ActionMask(mask_key='mask'),
            DoubleToFloat(),
            StepCounter(max_steps=200),
        ),
    )

In [4]:
env = env_maker()
check_env_specs(env)

2025-05-28 00:08:33,497 [torchrl][INFO] check_env_specs succeeded!


## Building the model (Deep Q-network)

The following function builds a :class:`~torchrl.modules.DuelingCnnDQNet`
object which is a simple CNN followed by a two-layer MLP. The only trick used
here is that the action values (i.e. left and right action value) are
computed using

\begin{align}\mathbb{v} = b(obs) + v(obs) - \mathbb{E}[v(obs)]\end{align}

where $\mathbb{v}$ is our vector of action values,
$b$ is a $\mathbb{R}^n \rightarrow 1$ function and $v$ is a
$\mathbb{R}^n \rightarrow \mathbb{R}^m$ function, for
$n = \# obs$ and $m = \# actions$.

Our network is wrapped in a :class:`~torchrl.modules.QValueActor`,
which will read the state-action
values, pick up the one with the maximum value and write all those results
in the input :class:`tensordict.TensorDict`.




In [5]:
q_net = MLP(in_channels=2, n_size=10, n_actions=100).to(device)

In [6]:
actor = QValueActor(
    module=q_net,
    spec=env.action_spec,
    action_mask_key='mask'
)

In [7]:
# total_frames = 5_000
total_frames = 2_000_000

eps_greedy_val = 0.1
eps_greedy_val_end = 0.005

In [8]:
exploration_module = EGreedyModule(
    spec=env.action_spec,
    annealing_num_steps=total_frames,
    eps_init=eps_greedy_val,
    eps_end=eps_greedy_val_end,
)

In [9]:
actor_explore = TensorDictSequential(actor, exploration_module)

## Collecting and storing data

### Replay buffers

Replay buffers play a central role in off-policy RL algorithms such as DQN.
They constitute the dataset we will be sampling from during training.

Here, we will use a regular sampling strategy, although a prioritized RB
could improve the performance significantly.

We place the storage on disk using
:class:`~torchrl.data.replay_buffers.storages.LazyMemmapStorage` class. This
storage is created in a lazy manner: it will only be instantiated once the
first batch of data is passed to it.

The only requirement of this storage is that the data passed to it at write
time must always have the same shape.



In [10]:
# batch_size = 32  # 256
batch_size = 1024  # 256
buffer_size = min(total_frames, 100000)
n_optim = 8

In [11]:
replay_buffer = TensorDictReplayBuffer(
    batch_size=batch_size,
    storage=LazyTensorStorage(max_size=buffer_size),
    prefetch=n_optim,
    transform=lambda td: td.to(device),
)

In [12]:
parallel_env = ParallelEnv(8, env_maker)

/home/timur/Coding/yandex_training_ml2.0/.venv/lib/python3.10/site-packages/gymnasium/envs/registration.py:644: UserWarning: WARN: Overriding environment Battleship-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [13]:
# frames_per_batch = 32  # 128
frames_per_batch = 512  # 128

In [14]:
collector = SyncDataCollector(
    create_env_fn=parallel_env,
    policy=actor_explore,
    total_frames=total_frames,
    frames_per_batch=frames_per_batch,
    reset_when_done=True,
    reset_at_each_iter=True,
    device=device,
    set_truncated=True,
)

/home/timur/Coding/yandex_training_ml2.0/.venv/lib/python3.10/site-packages/torchrl/collectors/collectors.py:835: UserWarning: total_frames (2000000) is not exactly divisible by frames_per_batch (512). This means 384 additional frames will be collected.To silence this message, set the environment variable RL_WARNINGS to False.
  warnings.warn(


## Loss function

Building our loss function is straightforward: we only need to provide
the model and a bunch of hyperparameters to the DQNLoss class.

### Target parameters

Many off-policy RL algorithms use the concept of "target parameters" when it
comes to estimate the value of the next state or state-action pair.
The target parameters are lagged copies of the model parameters. Because
their predictions mismatch those of the current model configuration, they
help learning by putting a pessimistic bound on the value being estimated.
This is a powerful trick (known as "Double Q-Learning") that is ubiquitous
in similar algorithms.




In [15]:
# gamma = 0.99
gamma = 0.7

In [16]:
loss_module = DQNLoss(actor, delay_value=True)
loss_module.make_value_estimator(gamma=gamma)
target_updater = SoftUpdate(loss_module, eps=0.995)

In [17]:
td = next(iter(collector)).reshape(-1)

In [104]:
td

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([512]), device=cuda:0, dtype=torch.int64, is_shared=True),
        action_value: Tensor(shape=torch.Size([512, 100]), device=cuda:0, dtype=torch.float32, is_shared=True),
        chosen_action_value: Tensor(shape=torch.Size([512, 1]), device=cuda:0, dtype=torch.float32, is_shared=True),
        collector: TensorDict(
            fields={
                traj_ids: Tensor(shape=torch.Size([512]), device=cuda:0, dtype=torch.int64, is_shared=True)},
            batch_size=torch.Size([512]),
            device=cuda:0,
            is_shared=True),
        done: Tensor(shape=torch.Size([512, 1]), device=cuda:0, dtype=torch.bool, is_shared=True),
        mask: Tensor(shape=torch.Size([512, 100]), device=cuda:0, dtype=torch.int8, is_shared=True),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([512, 1]), device=cuda:0, dtype=torch.bool, is_shared=True),
                mask: Tensor(shap

## Building a Trainer

TorchRL's :class:`~torchrl.trainers.Trainer` class constructor takes the
following keyword-only arguments:

- ``collector``
- ``loss_module``
- ``optimizer``
- ``logger``: A logger can be
- ``total_frames``: this parameter defines the lifespan of the trainer.
- ``frame_skip``: when a frame-skip is used, the collector must be made
  aware of it in order to accurately count the number of frames
  collected etc. Making the trainer aware of this parameter is not
  mandatory but helps to have a fairer comparison between settings where
  the total number of frames (budget) is fixed but the frame-skip is
  variable.



In [18]:
# the learning rate of the optimizer
lr = 3e-5
# weight decay
wd = 1e-5

In [19]:
optimizer = torch.optim.Adam(
    loss_module.parameters(),
    lr=lr,
    # weight_decay=wd
)

In [20]:
from torchrl.record.loggers.tensorboard import TensorboardLogger
import os

log_dir = 'runs/dqn_battleship'
existing = [d for d in os.listdir(log_dir) if os.path.isdir(os.path.join(log_dir, d))]
exp_no = len(existing) + 1
logger = TensorboardLogger(
    exp_name=f"exp_{exp_no}",
    log_dir=log_dir
)

print(exp_no)

9


We can control how often the scalars should be logged. Here we set this
to a low value as our training loop is short:



In [21]:
log_interval = 2048

trainer = Trainer(
    collector=collector,
    total_frames=total_frames,
    frame_skip=1,
    loss_module=loss_module,
    optimizer=optimizer,
    logger=logger,
    optim_steps_per_batch=n_optim,
    log_interval=log_interval,
    progress_bar=True
)

### Registering hooks

Registering hooks can be achieved in two separate ways:

- If the hook has it, the :meth:`~torchrl.trainers.TrainerHookBase.register`
  method is the first choice. One just needs to provide the trainer as input
  and the hook will be registered with a default name at a default location.
  For some hooks, the registration can be quite complex: :class:`~torchrl.trainers.ReplayBufferTrainer`
  requires 3 hooks (``extend``, ``sample`` and ``update_priority``) which
  can be cumbersome to implement.



In [ ]:
buffer_hook = ReplayBufferTrainer(
    replay_buffer=replay_buffer,
    flatten_tensordicts=True,
)
buffer_hook.register(trainer)
weight_updater = UpdateWeights(collector, update_weights_interval=1)
weight_updater.register(trainer)
recorder = LogValidationReward(
    record_interval=32,  # log every 100 optimization steps
    record_frames=1000,  # maximum number of frames in the record
    frame_skip=1,
    policy_exploration=actor_explore,
    environment=env,
    exploration_type=ExplorationType.DETERMINISTIC,
    log_keys=[("next", "reward")],
    out_keys={("next", "reward"): "rewards"},
    log_pbar=True,
)
recorder.register(trainer)

The exploration module epsilon factor is also annealed:




In [25]:
trainer.register_op("post_steps", actor_explore[1].step, frames=frames_per_batch)

- Any callable (including :class:`~torchrl.trainers.TrainerHookBase`
  subclasses) can be registered using :meth:`~torchrl.trainers.Trainer.register_op`.
  In this case, a location must be explicitly passed (). This method gives
  more control over the location of the hook but it also requires more
  understanding of the Trainer mechanism.
  Check the `trainer documentation <ref_trainers>`
  for a detailed description of the trainer hooks.




In [26]:
trainer.register_op("post_optim", target_updater.step)

We can log the training rewards too. Note that this is of limited interest
with CartPole, as rewards are always 1. The discounted sum of rewards is
maximised not by getting higher rewards but by keeping the cart-pole alive
for longer.
This will be reflected by the `total_rewards` value displayed in the
progress bar.




In [27]:
log_reward = LogScalar(log_pbar=True)
log_reward.register(trainer)

In [28]:
trainer.train()

  0%|          | 0/2000000 [00:00<?, ?it/s]/home/timur/Coding/yandex_training_ml2.0/.venv/lib/python3.10/site-packages/torchrl/envs/libs/gym.py:1150: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  action = int(action)
r_training: 0.2201, rewards: 0.2778, total_rewards: 12.6273: 100%|█████████▉| 1999872/2000000 [20:34<00:00, 592.22it/s] 

In [29]:
torch.save(q_net.state_dict(), f"checkpoints/dqn_{exp_no}.pt")
print(exp_no)

9


We can now quickly check the CSVs with the results.



In [ ]:
def print_csv_files_in_folder(folder_path):
    """
    Find all CSV files in a folder and prints the first 10 lines of each file.

    Args:
        folder_path (str): The relative path to the folder.

    """
    csv_files = []
    output_str = ""
    for dirpath, _, filenames in os.walk(folder_path):
        for file in filenames:
            if file.endswith(".csv"):
                csv_files.append(os.path.join(dirpath, file))
    for csv_file in csv_files:
        output_str += f"File: {csv_file}\n"
        with open(csv_file, "r") as f:
            for i, line in enumerate(f):
                if i == 10:
                    break
                output_str += line.strip() + "\n"
        output_str += "\n"
    print(output_str)


print_csv_files_in_folder(logger.experiment.log_dir)

trainer.shutdown()
del trainer

## Conclusion and possible improvements

In this tutorial we have learned:

- How to write a Trainer, including building its components and registering
  them in the trainer;
- How to code a DQN algorithm, including how to create a policy that picks
  up the action with the highest value with
  :class:`~torchrl.modules.QValueNetwork`;
- How to build a multiprocessed data collector;

Possible improvements to this tutorial could include:

- A prioritized replay buffer could also be used. This will give a
  higher priority to samples that have the worst value accuracy.
  Learn more on the
  `replay buffer section <ref_buffers>`
  of the documentation.
- A distributional loss (see :class:`~torchrl.objectives.DistributionalDQNLoss`
  for more information).
- More fancy exploration techniques, such as :class:`~torchrl.modules.NoisyLinear` layers and such.

